[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)

# Evaluate, Experiment, Improve

Notebook 2 made the agent's behavior visible. Now we will evaluate a few important skills,
inspect failures in Phoenix, and test a prompt change against the same examples.

**Learning path:** run a small baseline → inspect router and skill scores → change one prompt
→ compare the same cases → review a failure → decide whether it belongs in the golden dataset.

The notebook uploads all 11 frozen golden cases. The same dataset version and four named
evaluators are used for the baseline and prompt-change experiments.


---
## 1. Install the workshop and Phoenix client packages

This mirrors Notebook 2's Phoenix setup. The board is hosted, and the base packages let this
notebook run on its own Colab runtime too.


In [ ]:
%pip install -q "openai>=1.40" "pandas>=2.0" "matplotlib>=3.8" arize-phoenix-otel==0.17.1 arize-phoenix-client==3.7.0 openinference-instrumentation-openai==0.1.76

> Install before importing the agent or tracing SDK. If Colab says the runtime
> needs a restart, restart it and rerun the notebook from the top.


---
## 2. Load the same agent and credentials as Notebook 2

Notebook 3 imports the canonical agent from `utils.py`, loads the frozen golden cases, and
uses the same gateway and seat key as Notebook 2. The Phoenix board key is separate from the
workshop model key.


In [ ]:
import urllib.request
import os
import importlib
from pathlib import Path

utils_path = Path("utils.py")
required_utils = (
    "def set_span_tracer(tracer, attributes=None):",
    "def load_golden():",
    "def run_suite(label, system_prompt, temperature, agent_fn=None, judge_fn=None,",
)
if not utils_path.exists() or any(marker not in utils_path.read_text() for marker in required_utils):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils
utils = importlib.reload(utils)

utils.load_data()
utils.load_golden()
utils.connect()
SEAT = utils.seat_tag()
PHOENIX_BOARD_URL = "https://fmlds-eval-workshop-phoenix.fly.dev"
PHOENIX_EVAL_PROJECT = "01-Part-3-evaluations"
PHOENIX_API_KEY = utils.get_secret("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    from getpass import getpass
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
from phoenix.client import Client
phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
try:
    phoenix_client.projects.list()
except Exception as exc:
    if "401" not in str(exc) and "Unauthorized" not in str(exc):
        raise
    from getpass import getpass
    print("Saved Phoenix key was rejected. Enter the participant board key supplied by the organizer.")
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
    phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
    phoenix_client.projects.list()

from utils import (
    AGENT_MODEL, CREATIVE_SYSTEM_PROMPT, HARDENED_SYSTEM_PROMPT, GOLDEN,
    _case_passes, check_code, check_numeric, check_router, judge_case, run_agent,
)

print("bootstrap done |", len(GOLDEN["cases"]), "golden cases | seat:", SEAT)


## 3. Send this notebook's traces to Phoenix

As in Notebook 2, the OTLP exporter writes directly to the shared board. Notebook 3 uses
`01-Part-3-evaluations`, separate from Notebook 2's trace-review project. The stable
`session.id` identifies your seat; unique run labels distinguish experiment variants.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name=PHOENIX_EVAL_PROJECT,
    endpoint=f"{PHOENIX_BOARD_URL}/v1/traces",
    protocol="http/protobuf",
    batch=False,
    api_key=PHOENIX_API_KEY,
)
print("Phoenix project:", PHOENIX_EVAL_PROJECT)
print("Phoenix board:", PHOENIX_BOARD_URL)


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.semconv.trace import SpanAttributes

openai_instrumentor = OpenAIInstrumentor()
if openai_instrumentor.is_instrumented_by_opentelemetry:
    openai_instrumentor.uninstrument()
openai_instrumentor.instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(
    tracer_provider.get_tracer(__name__),
    attributes={SpanAttributes.SESSION_ID: SEAT},
)
print("OpenAI and agent spans are instrumented")


---
## 4. Create a Phoenix dataset

Phoenix Experiments use three parts: a **dataset** of fixed cases, a **task** that runs the
agent, and **evaluators** that score its outputs. Both prompt variants will run against this
same frozen 11-case golden dataset, so their scores appear in the dataset's **Experiments** tab.


In [ ]:
import datetime as _datetime
import json
import pandas as pd

EXPERIMENT_CASES = GOLDEN["cases"]
if not EXPERIMENT_CASES:
    raise ValueError("Golden dataset has no cases")

RUN_TAG = _datetime.datetime.now(_datetime.timezone.utc).strftime("lab3-%Y%m%d-%H%M%S-%f")
PHOENIX_DATASET_NAME = f"eval-workshop-golden-{SEAT}-{RUN_TAG}"
golden_experiment_df = pd.DataFrame([
    {
        "case_id": case["id"],
        "query": case["query"],
        "case_type": case["type"],
        "expected_json": json.dumps(case["expected"], sort_keys=True),
        "tolerance": float(case.get("tolerance", 0.0)),
        "judge_rubric_json": json.dumps(case.get("judge_rubric") or []),
        "ground_truth_logic": case.get("ground_truth_logic", ""),
        "trap": case.get("trap") or "",
    }
    for case in EXPERIMENT_CASES
])
experiment_dataset = phoenix_client.datasets.create_dataset(
    name=PHOENIX_DATASET_NAME,
    dataset_description="Frozen workshop golden cases for router, tool, answer, and overall evaluation.",
    dataframe=golden_experiment_df,
    input_keys=["case_id", "query"],
    output_keys=["case_type", "expected_json", "tolerance", "judge_rubric_json", "ground_truth_logic"],
    metadata_keys=["trap"],
)
print("Phoenix dataset:", PHOENIX_DATASET_NAME)
print("Dataset ID:", experiment_dataset.id, "| version:", experiment_dataset.version_id)
print("Golden examples uploaded:", len(golden_experiment_df))
print("Experiments tab:", phoenix_client.experiments.get_dataset_experiments_url(experiment_dataset.id))


### What are the evaluators checking?

Phoenix runs each named evaluator on each task output and reports its average score in the
experiment summary. We'll use separate code evaluators for the router, generated tool code, and
numeric answer, plus one strict overall evaluator that requires every applicable check to pass.

1. **Router evaluator** — did the agent call the required tool?
2. **Tool/code evaluator** — did the analysis tool generate valid code that ran successfully?
3. **Answer evaluator** — did the final answer match the frozen ground truth?

The overall evaluator also uses the LLM judge for the missing-location behavior case. A router
pass does not imply a tool or answer pass; the Experiments table will show them in separate
score columns.

The analysis tool also cautions the code generator to inspect the full dataset for likely
entity variants and decoys, using bounded summaries instead of dumping rows. The merchant case
tests whether that general caution is enough; it is not a guarantee, so inspect the generated
code and trace evidence.


In [ ]:
def case_from_example(input, expected, metadata):
    """Rebuild the golden case shape used by the shared code evaluators."""
    return {
        "id": input["case_id"],
        "query": input["query"],
        "type": expected["case_type"],
        "expected": json.loads(expected["expected_json"]),
        "tolerance": float(expected.get("tolerance", 0.0)),
        "judge_rubric": json.loads(expected.get("judge_rubric_json") or "[]"),
        "ground_truth_logic": expected.get("ground_truth_logic", ""),
        "trap": metadata.get("trap"),
    }


def make_agent_task(system_prompt, temperature, model, run_label):
    """Return a Phoenix task that runs the complete tool-using agent once per example."""
    def task(input):
        result = run_agent(
            input["query"],
            system_prompt=system_prompt,
            temperature=temperature,
            model=model,
            label=f"{run_label}/{input['case_id']}",
        )
        # Keep the experiment result focused; traces retain the full span details.
        assistant_messages = [
            {"role": "assistant", "tool_calls": message.get("tool_calls") or []}
            for message in result["messages"]
            if message.get("role") == "assistant"
        ]
        return {
            "case_id": input["case_id"],
            "answer": result["answer"],
            "messages": assistant_messages,
            "executed_code": result["executed_code"],
        }
    return task


#### Evaluator 1 — router choice

This check reads the actual tool calls from the agent run and compares them with the tool
required by the golden case. It does not judge whether the tool's result is correct. Phoenix
will aggregate these 0/1 scores under `router_eval`.


In [ ]:
def router_eval(input, output, expected, metadata):
    case = case_from_example(input, expected, metadata)
    result = check_router(case, output)
    correct = bool(result["router_ok"])
    return {
        "score": int(correct),
        "label": "correct" if correct else "incorrect",
        "explanation": f"Expected {REQUIRED_TOOLS.get(case['id'], [])}; used {result['tools_used']}.",
    }


#### Evaluator 2 — analysis tool code

For code-producing cases, this checks syntax and whether execution raised an error. It is a
tool/skill check, not an answer-correctness check. Non-code cases are marked not applicable.
Phoenix aggregates applicable scores under `tool_eval`.


In [ ]:
def tool_eval(input, output, expected, metadata):
    if not output.get("executed_code"):
        return {"score": None, "label": "not_applicable", "explanation": "No generated code for this case."}
    result = check_code(output)
    runnable = bool(result["code_valid"] and result["no_traceback"])
    return {
        "score": int(runnable),
        "label": "runnable" if runnable else "not_runnable",
        "explanation": f"Valid syntax={result['code_valid']}; execution succeeded={result['no_traceback']}.",
    }


#### Evaluator 3 — answer against ground truth

This compares the extracted `FINAL ANSWER` with the case's frozen numeric value. It is not
applicable to behavior-only cases. Phoenix aggregates numeric scores under `numeric_eval`.


In [ ]:
def numeric_eval(input, output, expected, metadata):
    case = case_from_example(input, expected, metadata)
    if case["type"] != "numeric":
        return {"score": None, "label": "not_applicable", "explanation": "No numeric target for this case."}
    result = check_numeric(case, output)
    correct = bool(result["numeric_ok"])
    return {
        "score": int(correct),
        "label": "correct" if correct else "incorrect",
        "explanation": f"Extracted={result['extracted']}; compared with frozen ground truth and tolerance.",
    }


#### Evaluator 4 — strict overall pass

The overall evaluator combines all applicable checks for one example. Numeric cases require
router, tool/code, and ground-truth checks. Behavior cases use the router and behavior judge;
generated code is checked if the agent produced any. Phoenix reports its average as `overall_eval`.


In [ ]:
def overall_eval(input, output, expected, metadata):
    case = case_from_example(input, expected, metadata)
    checks = {"answered": bool((output.get("answer") or "").strip())}
    checks.update(check_router(case, output))
    if case["type"] == "numeric" or (case["type"] == "behavior" and output.get("executed_code")):
        checks.update(check_code(output))
    if case["type"] == "numeric":
        checks.update(check_numeric(case, output))
    if case["type"] in ("behavior", "advisory"):
        verdict = judge_case(case, output)
        checks["judge_pass"] = bool(verdict.get("pass"))
        checks["judge_reason"] = verdict.get("reason", "")
    passed = bool(_case_passes(case, checks))
    scored_checks = (
        "answered", "router_ok", "code_valid", "no_traceback", "numeric_ok",
        "category_ok", "merchant_ok", "judge_pass",
    )
    failed_checks = [key for key in scored_checks if key in checks and not checks[key]]
    explanation = "All required checks passed." if passed else f"Failed checks: {failed_checks or ['judge_pass']}"
    return {"score": int(passed), "label": "pass" if passed else "fail", "explanation": explanation}


EXPERIMENT_EVALUATORS = {
    "router_eval": router_eval,
    "tool_eval": tool_eval,
    "numeric_eval": numeric_eval,
    "overall_eval": overall_eval,
}
print("Phoenix evaluators:", list(EXPERIMENT_EVALUATORS))


### Run the hardened baseline in Phoenix

The Phoenix client creates a native dataset experiment. Its summary reports each evaluator's
average score, and the Experiments tab stores per-example inputs, outputs, labels, scores, and
linked traces.


In [ ]:
BASELINE_EXPERIMENT_NAME = f"{RUN_TAG}-prompt-baseline"
baseline_experiment = phoenix_client.experiments.run_experiment(
    dataset=experiment_dataset,
    task=make_agent_task(HARDENED_SYSTEM_PROMPT, 0.0, AGENT_MODEL, BASELINE_EXPERIMENT_NAME),
    evaluators=EXPERIMENT_EVALUATORS,
    experiment_name=BASELINE_EXPERIMENT_NAME,
    experiment_description="Hardened prompt on the frozen 11-case golden dataset.",
    experiment_metadata={"seat": SEAT, "prompt_variant": "hardened", "model": AGENT_MODEL, "temperature": 0.0},
    print_summary=True,
)


#### Inspect the three code checks on one saved run

Phoenix already ran these evaluators across the dataset. These separate cells demonstrate the
underlying checks on the baseline's saved October-expenses result; they do not make additional
agent calls.


In [ ]:
baseline_outputs = {
    run["output"]["case_id"]: run["output"]
    for run in baseline_experiment["task_runs"]
    if run.get("output") and run["output"].get("case_id")
}
example_case = next(c for c in EXPERIMENT_CASES if c["id"] == "total_expenses_october")
if example_case["id"] not in baseline_outputs:
    raise RuntimeError("The baseline did not return the example task output; inspect experiment errors.")
example_result = baseline_outputs[example_case["id"]]


In [ ]:
router_result = check_router(example_case, example_result)
router_result["label"] = "correct" if router_result["router_ok"] else "incorrect"
router_result["score"] = int(router_result["router_ok"])
router_result


In [ ]:
tool_result = check_code(example_result)
tool_ok = bool(tool_result["code_valid"] and tool_result["no_traceback"])
tool_result["label"] = "runnable" if tool_ok else "not_runnable"
tool_result["score"] = int(tool_ok)
tool_result


In [ ]:
numeric_result = check_numeric(example_case, example_result)
numeric_result["label"] = "correct" if numeric_result["numeric_ok"] else "incorrect"
numeric_result["score"] = int(numeric_result["numeric_ok"])
numeric_result


---
## 5. Run a prompt experiment

An experiment changes one thing and holds the rest fixed: **same questions, model,
temperature, and evaluators**. The variant below intentionally relaxes tool-use and
precision guidance. Run it, compare the scores, then inspect the corresponding traces.

This creates a second Phoenix experiment on the **same dataset version** as the baseline. The
dataset's Experiments tab will show one score column per evaluator, including the overall pass
evaluator, and let you compare individual cases and traces side by side.


In [ ]:
CANDIDATE_SYSTEM_PROMPT = CREATIVE_SYSTEM_PROMPT
CANDIDATE_EXPERIMENT_NAME = f"{RUN_TAG}-prompt-candidate"
print("Candidate system prompt:\n", CANDIDATE_SYSTEM_PROMPT)

candidate_experiment = phoenix_client.experiments.run_experiment(
    dataset=experiment_dataset,
    task=make_agent_task(CANDIDATE_SYSTEM_PROMPT, 0.0, AGENT_MODEL, CANDIDATE_EXPERIMENT_NAME),
    evaluators=EXPERIMENT_EVALUATORS,
    experiment_name=CANDIDATE_EXPERIMENT_NAME,
    experiment_description="Creative prompt variant on the baseline dataset version.",
    experiment_metadata={"seat": SEAT, "prompt_variant": "creative", "model": AGENT_MODEL, "temperature": 0.0},
    print_summary=True,
)


### Compare baseline and candidate in Phoenix

Open the dataset's **Experiments** tab and compare `prompt-baseline` with `prompt-candidate`.
Each evaluator has its own average-score column; `overall_eval` is the fraction of examples that
passed every required check. Open a low-scoring row to read the task output, evaluator
explanations, and linked trace.


### Optional: compare two router models instead

Keep the hardened prompt, dataset, and evaluators fixed, then change only the router model. The
workshop gateway is configured for `gpt-4o-mini` and `gpt-4o`; the larger model may use more
of your seat's budget. This agent also makes a separate code-generation call inside its data
tool; that inner call stays on `gpt-4o-mini`, so this is a router-model comparison, not a
whole-agent model comparison. This optional comparison uses the same golden dataset and
evaluators, so its runs also appear in the dataset's Experiments tab. Set the flag in the next
cell to `True` to run the two additional full-suite experiments. Check the gateway's currently
enabled model aliases before changing these names.


In [ ]:
MODEL_A = AGENT_MODEL
MODEL_B = "gpt-4o"
model_a_name = f"{RUN_TAG}-model-{MODEL_A}"
model_b_name = f"{RUN_TAG}-model-{MODEL_B}"

RUN_MODEL_COMPARISON = False  # Set True to run two additional full-suite experiments.
if RUN_MODEL_COMPARISON:
    model_a_experiment = phoenix_client.experiments.run_experiment(
        dataset=experiment_dataset,
        task=make_agent_task(HARDENED_SYSTEM_PROMPT, 0.0, MODEL_A, model_a_name),
        evaluators=EXPERIMENT_EVALUATORS,
        experiment_name=model_a_name,
        experiment_description=f"Router model comparison using {MODEL_A}.",
        experiment_metadata={"seat": SEAT, "prompt_variant": "hardened", "model": MODEL_A},
        print_summary=True,
    )
    model_b_experiment = phoenix_client.experiments.run_experiment(
        dataset=experiment_dataset,
        task=make_agent_task(HARDENED_SYSTEM_PROMPT, 0.0, MODEL_B, model_b_name),
        evaluators=EXPERIMENT_EVALUATORS,
        experiment_name=model_b_name,
        experiment_description=f"Router model comparison using {MODEL_B}.",
        experiment_metadata={"seat": SEAT, "prompt_variant": "hardened", "model": MODEL_B},
        print_summary=True,
    )
else:
    print("Skipping optional model comparison. Set RUN_MODEL_COMPARISON=True to run it.")


---
## 6. Inspect a failed experiment example in Phoenix

Open this dataset's **Experiments** tab, select the candidate experiment, and filter for a low
`overall_eval` score. Open the example run and compare the query, task output, individual
evaluator labels/explanations, and linked trace. Which component failed?

Use the trace to decide whether this is a real agent failure or an evaluator/ground-truth issue.
Record evidence before promoting anything into the golden dataset.


In [ ]:
print("Board:", PHOENIX_BOARD_URL)
print("Dataset:", PHOENIX_DATASET_NAME, "|", experiment_dataset.id)
print("Experiments tab:", phoenix_client.experiments.get_dataset_experiments_url(experiment_dataset.id))
print("Baseline:", BASELINE_EXPERIMENT_NAME)
print("Candidate:", CANDIDATE_EXPERIMENT_NAME)
print("Candidate comparison link:", phoenix_client.experiments.get_experiment_url(
    experiment_dataset.id, candidate_experiment["experiment_id"]
))


In [ ]:
# Complete this after inspecting the trace. Keep the evidence concise and reproducible.
failure_note = {
    "experiment": CANDIDATE_EXPERIMENT_NAME,
    "case_id": "replace-with-failed-case-id",
    "failure_type": "router | skill | data | evaluator | prompt",
    "trace_evidence": "Which experiment output/evaluator/trace span shows the failure?",
    "expected_behavior": "What should the agent have done?",
    "reproducible": "yes | no | not_checked",
}
failure_note


---
## 7. Promote a reviewed failure to the golden dataset

Only promote a case when the failure is reproducible, the expected behavior is clear, and
ground truth can be independently checked. Capture at least: a unique `id`, the exact user
`query`, case `type`, expected behavior/value, tolerance (for numeric cases), and the rule for
computing ground truth.

For this repository, golden cases are generated by `scripts/generate_dataset.py`; don't edit
the JSON by hand. Add the case to `build_cases()` and compute its expected value from the
synthetic data, then regenerate and verify with `python scripts/generate_dataset.py --check`.
The notebook note above is a review candidate, not an automatic dataset change.


## Phoenix has saved the experiment results

The SDK records the dataset version, task outputs, named evaluator scores, overall score, and
task traces as Phoenix experiment results. No separate annotation-upload step is needed. Use the
dataset's **Experiments** tab for aggregate scores and side-by-side examples; open an example
run's trace to inspect the router, tool, and model calls that produced it.


---
## Takeaways

- Router, execution, answer-quality, and ground-truth checks catch different failures.
- A prompt or model change is an experiment only when you compare it on the same cases with
  the same evaluators and hold other variables fixed.
- Phoenix's dataset **Experiments** tab shows the per-evaluator averages and overall score;
  traces help explain *why* a score changed.
- Human review decides whether a failure is real and worth adding to the golden set.
- The complete golden suite and regression gate remain in `scripts/run_evals.py` and CI.


---
### Workshop home

[Return to the workshop README](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/README.md)
